# 02 — Periodicity Features (Lomb-Scargle)

Run Lomb-Scargle period-finding on all objects, phase-fold examples,
and merge periodicity features into the full feature table.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG, FEATURE_CONFIG
from src.data import get_object_lightcurve, get_objects_by_class
from src.features import compute_lomb_scargle
from src.utils import get_class_name, save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
metadata = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet'))
lc = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_lc.parquet'))
features = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'features_statistical.parquet'))
print(f"Feature table: {features.shape}")

## 1. Lomb-Scargle Demo — Known Periodic Source

Demonstrate on an RR Lyrae (class 92), which should show a strong periodic signal.

In [ ]:
# Pick an RR Lyrae
rrl_id = get_objects_by_class(metadata, 92, n=1)[0]
rrl_lc = get_object_lightcurve(rrl_id, lc)

# Run LS on g-band
if 1 in rrl_lc:  # g-band = passband 1
    g_lc = rrl_lc[1]
    ls_result = compute_lomb_scargle(g_lc['mjd'], g_lc['flux'], g_lc['flux_err'])
    print(f"RR Lyrae {rrl_id} — g-band Lomb-Scargle:")
    for k, v in ls_result.items():
        print(f"  {k}: {v}")

    # Plot periodogram
    min_freq = 1.0 / FEATURE_CONFIG['ls_max_period']
    max_freq = 1.0 / FEATURE_CONFIG['ls_min_period']
    frequency = np.linspace(min_freq, max_freq, FEATURE_CONFIG['ls_n_frequencies'])
    ls = LombScargle(g_lc['mjd'], g_lc['flux'], g_lc['flux_err'])
    power = ls.power(frequency)
    periods = 1.0 / frequency

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(periods, power, 'k-', linewidth=0.5)
    axes[0].axvline(ls_result['best_period'], color='red', linestyle='--',
                    label=f"Best: {ls_result['best_period']:.4f} d")
    axes[0].set_xlabel('Period (days)')
    axes[0].set_ylabel('Lomb-Scargle Power')
    axes[0].set_title(f'Periodogram — RR Lyrae {rrl_id}')
    axes[0].set_xscale('log')
    axes[0].legend()

    # Phase-folded light curve
    phase = (g_lc['mjd'] % ls_result['best_period']) / ls_result['best_period']
    axes[1].errorbar(phase, g_lc['flux'], yerr=g_lc['flux_err'],
                     fmt='o', markersize=4, alpha=0.7)
    axes[1].errorbar(phase + 1, g_lc['flux'], yerr=g_lc['flux_err'],
                     fmt='o', markersize=4, alpha=0.3, color='gray')
    axes[1].set_xlabel('Phase')
    axes[1].set_ylabel('Flux')
    axes[1].set_title(f'Phase-Folded (P = {ls_result["best_period"]:.4f} d)')
    axes[1].set_xlim(0, 2)

    plt.tight_layout()
    save_figure(fig, 'lomb_scargle_example')
    plt.show()

## 2. Best Period Distribution by Class

The statistical features already include Lomb-Scargle results (computed in the
previous notebook via `extract_per_band_features`). Let's analyze them.

In [ ]:
# Check which LS columns exist from the statistical features
ls_cols = [c for c in features.columns if 'best_period' in c or 'best_power' in c or 'fap' in c]
print(f"Lomb-Scargle columns: {ls_cols[:10]}")

In [ ]:
# Best period in g-band by class
if 'g_best_period' in features.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    for target in sorted(DATA_CONFIG['class_names'].keys()):
        subset = features[features['target'] == target]['g_best_period'].dropna()
        subset = subset[(subset > 0.1) & (subset < 500)]
        if len(subset) > 5:
            ax.hist(np.log10(subset), bins=30, alpha=0.4,
                    label=get_class_name(target), density=True)
    ax.set_xlabel('log10(Best Period) [days]')
    ax.set_ylabel('Density')
    ax.set_title('Lomb-Scargle Best Period by Class (g-band)')
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    save_figure(fig, 'period_by_class')
    plt.show()

In [ ]:
# FAP distribution — low FAP = real periodicity
if 'g_fap' in features.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    periodic_classes = [92, 53, 16]  # RR Lyrae, Mira, EB
    other_classes = [90, 42, 88]     # SN Ia, SN II, AGN

    for target in periodic_classes:
        fap = features[features['target'] == target]['g_fap'].dropna()
        fap = fap[fap > 0]
        if len(fap) > 0:
            ax.hist(np.log10(fap.clip(1e-30)), bins=30, alpha=0.5,
                    label=f"{get_class_name(target)} (periodic)", density=True)

    for target in other_classes:
        fap = features[features['target'] == target]['g_fap'].dropna()
        fap = fap[fap > 0]
        if len(fap) > 0:
            ax.hist(np.log10(fap.clip(1e-30)), bins=30, alpha=0.3,
                    label=f"{get_class_name(target)} (non-periodic)",
                    density=True, linestyle='--')

    ax.set_xlabel('log10(False Alarm Probability)')
    ax.set_ylabel('Density')
    ax.set_title('LS False Alarm Probability — Periodic vs Non-Periodic Classes')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 3. Save Final Feature Table

The statistical features table already includes Lomb-Scargle features from
`extract_per_band_features`. Save the final merged version.

In [ ]:
out_path = os.path.join(DATA_CONFIG['processed_dir'], 'features_all.parquet')
features.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)")
print(f"Shape: {features.shape}")